# Fig. S1G: exemplar-gene APA consistency across the 6 batches (mice)

Per-batch (biological-replicate) view of the two Fig 1e exemplar genes' UTR-site proportion
changes and W(t) trajectory over pseudotime, to check that the APA shifts are consistent between
the 6 independent batches. Uses raw (unsmoothed) counts so each batch is independent. Both
the equal-width and equal-cell pseudotime binnings are shown.

- Kernel: apa-python. Input: UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad.
- Output: $M26_RESULTS_ROOT/Fig1/supp_8_exemplars_per_mouse\_\_\*/.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import rankdata

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

PANEL_PREFIX, FIGURE = "supp_8_exemplars_per_mouse", "Fig1"
N_BINS = 20
BASE = Path(f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling")
RAW_PATH = BASE / "UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad"

# Raw (unsmoothed) counts -> per-batch proportions are independent across the 6
# batches (biological replicates); the graph-smoothed layer blends neighbours across batches
# and would make a cross-batch "consistency" claim circular, so we use raw counts here.
adata = sc.read_h5ad(RAW_PATH)
pt_all = adata.obs["dpt_pseudotime"].values.astype(float)
batch_all = adata.obs["batch_number"].astype(str).values
BATCHES = sorted(np.unique(batch_all), key=int)
EXEMPLARS = {"Hnrnpa2b1": "Shortening", "Qk": "Lengthening"}
print(f"raw counts: {adata.shape} | batches: {BATCHES}")

In [ ]:
def gene_props(gene_name):
    """Per-cell within-gene site proportions (sites sorted proximal->distal by utr_size)."""
    m = (adata.var["gene_name"] == gene_name).values
    sub = adata[:, m]
    X = sub.X
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    X = np.asarray(X, dtype=float)
    us = sub.var["utr_size"].values.astype(float)
    order = np.argsort(us)
    us = us[order]
    X = X[:, order]
    names = [sub.var["UTR_site"].values[i] for i in order]
    tot = X.sum(1)
    valid = tot > 0
    props = np.full_like(X, np.nan)
    props[valid] = X[valid] / tot[valid, None]
    return props, us, names, valid


def ecdf(x):
    """Empirical-CDF (rank/N) transform within the non-NaN values of x."""
    u = np.full_like(x, np.nan, dtype=float)
    v = ~np.isnan(x)
    u[v] = (rankdata(x[v], method="average") - 0.5) / v.sum()
    return u


def coord(inb, strategy):
    """x-coordinate per cell for a batch mask: raw pseudotime (equal-width) or the
    within-batch ECDF of pseudotime (equal-cell)."""
    if strategy == "equal-width":
        return np.where(inb, pt_all, np.nan)
    xc = np.full_like(pt_all, np.nan)
    xc[inb] = ecdf(pt_all[inb])
    return xc


def bin_means(xc, props, valid, n_bins=N_BINS):
    v = valid & ~np.isnan(xc)
    xv, pv = xc[v], props[v]
    if v.sum() < n_bins:
        n_bins = max(3, int(v.sum()) // 3)
    if n_bins < 1:
        return np.array([]), np.zeros((0, props.shape[1]))
    bins = pd.cut(xv, bins=n_bins, labels=False)
    cen, mp = [], []
    for b in range(n_bins):
        mm = bins == b
        if mm.sum() > 0:
            cen.append(xv[mm].mean())
            mp.append(pv[mm].mean(0))
    mp = np.array(mp)
    cen = np.array(cen)
    rs = mp.sum(1, keepdims=True)
    rs[rs == 0] = 1
    return cen, mp / rs


def W_ranks(us):
    K = len(us)
    return (rankdata(us, method="average") - 1) / (K - 1) if K > 1 else np.array([0.5])


def utr_cmap(k):
    return (
        ["#E24A33", "#4878CF"][:k] if k <= 2 else list(sns.color_palette("RdYlBu_r", k))
    )

### Per-batch UTR-site proportions (stacked area), one figure per gene x binning strategy


In [ ]:
# Per-batch stacked-area UTR-site proportions (one figure per gene x strategy).
for strategy in ["equal-width", "equal-cell"]:
    for gene, direction in EXEMPLARS.items():
        props, us, names, valid = gene_props(gene)
        K = len(us)
        colors = utr_cmap(K)
        fig, axes = plt.subplots(
            2,
            3,
            figsize=(S.FULL_W_PX / 300, S.mm_to_px(72) / 300),
            sharex=True,
            sharey=True,
        )
        axes = axes.ravel()
        for bi, b in enumerate(BATCHES):
            ax = axes[bi]
            inb = batch_all == b
            cen, mp = bin_means(coord(inb, strategy), props, valid & inb)
            if len(cen):
                ax.stackplot(cen, mp.T, colors=colors, alpha=0.85)
                ax.set_xlim(cen.min(), cen.max())
            ax.set_ylim(0, 1)
            ax.set_title(
                f"batch {b}  (n={int((inb & valid).sum())})", fontsize=S.FONT_TICK
            )
            if bi % 3 == 0:
                ax.set_ylabel("Proportion", fontsize=S.FONT_LABEL)
            if bi >= 3:
                ax.set_xlabel(
                    "Pseudotime"
                    + (" (equal-cell)" if strategy == "equal-cell" else ""),
                    fontsize=S.FONT_TICK,
                )
        for j in range(len(BATCHES), len(axes)):
            axes[j].axis("off")
        dc = S.COL_SHORT if direction == "Shortening" else S.COL_LONG
        fig.suptitle(
            f"{gene} ({direction}) - UTR-site proportions per batch [{strategy}]",
            fontsize=S.FONT_LABEL,
            color=dc,
            fontweight="bold",
        )
        fig.legend(
            [plt.Rectangle((0, 0), 1, 1, fc=colors[i]) for i in range(K)],
            names,
            loc="lower center",
            ncol=K,
            fontsize=S.FONT_LEGEND - 0.5,
            frameon=False,
            bbox_to_anchor=(0.5, -0.03),
        )
        fig.tight_layout(rect=[0, 0.04, 1, 0.95])
        S.save_and_show(
            fig,
            f"{PANEL_PREFIX}__props_{gene}_{strategy.replace('-', '')}",
            figure=FIGURE,
        )

### Per-batch UTR-site proportions, smoothed (equal-cell, Gaussian kernel)

Kernel-smoothed version of the equal-cell panels above: overlapping Gaussian windows in ECDF
(equal-cell) space instead of 20 disjoint bins, for a cleaner per-batch consistency read (same
smoothing as the Fig 2 organoid per-replicate panels). One figure per gene.

In [ ]:
# --- Per-batch UTR-site proportions, smoothed over the equal-cell (ECDF) pseudotime axis ---
# Gaussian kernel smoother in ECDF space (overlapping windows) instead of disjoint equal-cell
# bins -> smoother line, still uniform in cell density. Reuses gene_props / coord / utr_cmap /
# EXEMPLARS / BATCHES / batch_all from above.
SMOOTH_H = 0.12


def kernel_smooth(xc, props, valid, h=SMOOTH_H, n_eval=120):
    v = valid & ~np.isnan(xc); uu = xc[v]; pv = props[v]
    xe = np.linspace(0.02, 0.98, n_eval); out = np.zeros((n_eval, pv.shape[1]))
    for i, x in enumerate(xe):
        w = np.exp(-0.5 * ((uu - x) / h) ** 2); sw = w.sum()
        if sw > 0:
            out[i] = (w[:, None] * pv).sum(0) / sw
    rs = out.sum(1, keepdims=True); rs[rs == 0] = 1
    return xe, out / rs


for gene, direction in EXEMPLARS.items():
    props, us, names, valid = gene_props(gene)
    K = len(us); colors = utr_cmap(K)
    fig, axes = plt.subplots(2, 3, figsize=(S.FULL_W_PX / 300, S.mm_to_px(72) / 300),
                             sharex=True, sharey=True)
    axes = axes.ravel()
    for bi, b in enumerate(BATCHES):
        ax = axes[bi]; inb = batch_all == b
        cen, mp = kernel_smooth(coord(inb, "equal-cell"), props, valid & inb)
        if len(cen):
            ax.stackplot(cen, mp.T, colors=colors, alpha=0.85); ax.set_xlim(cen.min(), cen.max())
        ax.set_ylim(0, 1)
        ax.set_title(f"batch {b}  (n={int((inb & valid).sum())})", fontsize=S.FONT_TICK)
        if bi % 3 == 0:
            ax.set_ylabel("Proportion", fontsize=S.FONT_LABEL)
        if bi >= 3:
            ax.set_xlabel("Pseudotime (equal-cell)", fontsize=S.FONT_TICK)
    for j in range(len(BATCHES), len(axes)):
        axes[j].axis("off")
    dc = S.COL_SHORT if direction == "Shortening" else S.COL_LONG
    fig.suptitle(f"{gene} ({direction}) - UTR-site proportions per batch [equal-cell, smoothed h={SMOOTH_H}]",
                 fontsize=S.FONT_LABEL, color=dc, fontweight="bold")
    fig.legend([plt.Rectangle((0, 0), 1, 1, fc=colors[i]) for i in range(K)], names,
               loc="lower center", ncol=K, fontsize=S.FONT_LEGEND - 0.5, frameon=False,
               bbox_to_anchor=(0.5, -0.03))
    fig.tight_layout(rect=[0, 0.04, 1, 0.95])
    S.save_and_show(fig, f"{PANEL_PREFIX}__props_{gene}_equalcell_smooth", figure=FIGURE)


### Per-batch UTR proportions, equal-cell axis with cell-type boxes (mimics the main Fig 1e uniform panel)

Matches the main panel's left subpanel: equal-cell (ECDF) pseudotime binning (40 bins, piecewise-
linear - not kernel-smoothed, so the fast drop over the sparse aNSC intermediates is preserved),
with a cell-type-majority colour bar as the x-axis instead of pseudotime tick labels. (The
`_smooth` version used a Gaussian kernel h=0.12 that averaged over ~12% of cells and washed out
that fast drop.) Saved as `..._props_{gene}_equalcell_ctbar`.

In [ ]:
# --- Per-batch equal-cell proportions with cell-type boxes (mimics main Fig 1e uniform panel) ---
# 40 equal-cell bins (piecewise-linear, like the main panel's _bin_pseudotime; not kernel-smoothed)
# so the fast transition over the sparse aNSC intermediates is preserved; x-axis is a per-batch
# cell-type-majority colour bar (no pseudotime ticks). Reuses gene_props / coord / utr_cmap /
# EXEMPLARS / BATCHES / batch_all, and `adata` (for cell types) from above.
from matplotlib.patches import Patch as _Patch
from matplotlib.lines import Line2D as _L2D

_ct_all = adata.obs["celltype"].astype(str).values
_CT = S.MOUSE_LINEAGE_ORDER

for gene, direction in EXEMPLARS.items():
    props, us, names, valid = gene_props(gene)
    K = len(us); colors = utr_cmap(K)
    fig, axes = plt.subplots(2, 3, figsize=(S.FULL_W_PX / 300, S.mm_to_px(74) / 300),
                             sharex=True, sharey=True)
    axes = axes.ravel()
    for bi, b in enumerate(BATCHES):
        ax = axes[bi]; inb = batch_all == b
        xc = coord(inb, "equal-cell")                       # within-batch ECDF (equal-cell)
        cen, mp = bin_means(xc, props, valid & inb, n_bins=40)
        if len(cen):
            ax.stackplot(cen, mp.T, colors=colors, alpha=0.85)
        ax.set_ylim(-0.14, 1.0); ax.set_xlim(0, 1)
        for ct in _CT:                                      # cell-type boxes on the equal-cell axis
            cu = xc[inb & (_ct_all == ct)]; cu = cu[~np.isnan(cu)]
            if len(cu):
                ax.fill_between([cu.min(), cu.max()], -0.13, -0.02,
                                color=S.paper_palette_celltypes[ct], alpha=0.9, linewidth=0)
        ax.axhline(0, color="grey", lw=0.4, alpha=0.3)
        ax.set_title(f"batch {b}  (n={int((inb & valid).sum())})", fontsize=S.FONT_TICK)
        ax.set_xticks([]); ax.tick_params(axis="x", bottom=False, labelbottom=False)
        ax.spines["bottom"].set_visible(False)
        if bi % 3 == 0:
            ax.set_ylabel("Proportion", fontsize=S.FONT_LABEL)
    dc = S.COL_SHORT if direction == "Shortening" else S.COL_LONG
    fig.suptitle(f"{gene} ({direction}) - UTR-site proportions per batch "
                 f"[equal-cell; axis bar = majority cell type]",
                 fontsize=S.FONT_LABEL, color=dc, fontweight="bold")
    present = [c for c in _CT if c in set(_ct_all)]
    utr_h = [_Patch(facecolor=colors[i], label=names[i]) for i in range(K)]
    ct_h = [_L2D([0], [0], marker="s", color="w", markerfacecolor=S.paper_palette_celltypes[c],
                 markersize=5, label=c) for c in present]
    leg1 = fig.legend(handles=utr_h, loc="lower center", ncol=K, fontsize=S.FONT_LEGEND - 0.5,
                      frameon=False, bbox_to_anchor=(0.5, 0.045), title="UTR site",
                      title_fontsize=S.FONT_LEGEND - 0.5)
    fig.add_artist(leg1)
    fig.legend(handles=ct_h, loc="lower center", ncol=len(present), fontsize=S.FONT_LEGEND - 0.5,
               frameon=False, bbox_to_anchor=(0.5, -0.02), title="cell type (axis bar)",
               title_fontsize=S.FONT_LEGEND - 0.5)
    fig.tight_layout(rect=[0, 0.10, 1, 0.95])
    S.save_and_show(fig, f"{PANEL_PREFIX}__props_{gene}_equalcell_ctbar", figure=FIGURE)


### Per-batch W(t) overlay - the direct cross-replicate consistency view


In [ ]:
# W(t) trajectory per batch, overlaid (the clearest cross-replicate consistency view):
# 2 rows (equal-width / equal-cell) x 2 genes, each with the 6 batch curves on one axis.
fig, axes = plt.subplots(2, 2, figsize=(S.FULL_W_PX / 300, S.mm_to_px(95) / 300))
batch_pal = dict(zip(BATCHES, plt.cm.tab10(np.linspace(0, 1, 10))))
for row, strategy in enumerate(["equal-width", "equal-cell"]):
    for col, (gene, direction) in enumerate(EXEMPLARS.items()):
        ax = axes[row, col]
        props, us, names, valid = gene_props(gene)
        ranks = W_ranks(us)
        for b in BATCHES:
            inb = batch_all == b
            xc = coord(inb, strategy)
            v = valid & inb & ~np.isnan(xc)
            if v.sum() < 10:
                continue
            xv, pv = xc[v], props[v]
            bins = pd.cut(xv, bins=N_BINS, labels=False)
            bc, wm = [], []
            for bb in range(N_BINS):
                mm = bins == bb
                if mm.sum() > 0:
                    bc.append(xv[mm].mean())
                    wm.append(float((pv[mm] @ ranks).mean()))
            ax.plot(
                bc, wm, "-", color=batch_pal[b], lw=1.0, alpha=0.85, label=f"batch {b}"
            )
        dc = S.COL_SHORT if direction == "Shortening" else S.COL_LONG
        ax.set_title(
            f"{gene} ({direction})  [{strategy}]",
            fontsize=S.FONT_TICK,
            color=dc,
            fontweight="bold",
        )
        ax.set_ylabel("W(t)", fontsize=S.FONT_LABEL)
        ax.set_xlabel(
            "Pseudotime" + (" (equal-cell)" if strategy == "equal-cell" else ""),
            fontsize=S.FONT_TICK,
        )
        if row == 0 and col == 1:
            ax.legend(
                fontsize=S.FONT_LEGEND - 1,
                ncol=2,
                frameon=False,
                title="batch",
                title_fontsize=S.FONT_LEGEND - 1,
            )
fig.suptitle(
    "Per-batch W(t) consistency (6 biological replicates)", fontsize=S.FONT_LABEL
)
fig.tight_layout(rect=[0, 0, 1, 0.96])
S.save_and_show(fig, f"{PANEL_PREFIX}__W_overlay", figure=FIGURE)